# AELIONIX BLACKFORGE — Phase 8 Colab Validation

This notebook performs a deterministic, one-click validation of the **Business
Logic & Attack-Path Capability Foundation**.

It exercises the full `blackforge.business_logic` pipeline on the mock shop:

* **workflow_discovery / workflow_modeling** — the order lifecycle model
* **state_transition_analysis** — the anomalous `created -> shipped` path
* **business_rule_analysis** and **workflow_consistency_analysis** — invariants
* **ownership_analysis / role_boundary_analysis** — explicit test identities only
* **controlled_workflow_replay** — deterministic, safety-gated paper replay
* **business_logic_hypothesis / business_logic_validation** — hypothesis
  elevation with fail-closed gating
* **workflow_evidence_collection** — deterministic evidence for attribution

Every capability runs the same guarded pipeline: request validation, scope /
authorization, explicit-identity enforcement, fail-closed replay pre-check,
mock transport, normalization, evidence persistence, and world-model
materialization. **No free-form execution, no credential use, and no
autonomous identity discovery are possible through this surface.**

> Run all cells top-to-bottom. No GPU, no external services, no credentials.
> The notebook fails loudly on any check.

---

In [ ]:
import sys
import platform

print("Blackforge Phase 8 Colab Validation (Business Logic & Attack-Path Capability Foundation)")
print("=" * 60)
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Architecture:", platform.machine())
print("=" * 60)

assert sys.version_info >= (3, 10), f"Blackforge requires Python 3.10+, got {sys.version}"
print("Python version check: PASS")

---

In [ ]:
from pathlib import Path
import subprocess
import sys
import os
import shutil

# -- Configuration (edit here if fork changes) ---------------------------
REPO_URL = "https://github.com/Sagelord00000001/Blackforge.git"
REPO_DIR = Path("/content/blackforge")
# -----------------------------------------------------------------------

if REPO_DIR.exists() and (REPO_DIR / "blackforge" / "__init__.py").exists():
    print(f"Repository already exists at {REPO_DIR}, updating...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(str(REPO_DIR))
print(f"Repository ready at {REPO_DIR}")

---

In [ ]:
import subprocess

try:
    commit = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    print("Commit:", commit)
except Exception as e:
    print("Commit unavailable (expected in scratch checkouts):", e)

---

In [ ]:
!pip install hatchling --quiet
!pip install -e ".[dev]" --quiet

---

In [ ]:
import importlib

modules = [
    "blackforge",
    "blackforge.core.config",
    "blackforge.core.errors",
    "blackforge.core.types",
    "blackforge.runtime.bootstrap",
    "blackforge.memory",
    "blackforge.evidence",
    "blackforge.world_model",
    "blackforge.world_model.models",
    "blackforge.world_model.canonical",
    "blackforge.world_model.rules",
    "blackforge.world_model.query",
    "blackforge.world_model.repository",
    "blackforge.world_model.store",
    "blackforge.world_model.materializer",
    "blackforge.mission.manager",
    "blackforge.capabilities.registry",
    "blackforge.authorization",
    "blackforge.scope.models",
    "blackforge.scope.validator",
    "blackforge.recon",
    "blackforge.recon.models",
    "blackforge.recon.mock",
    "blackforge.recon.normalization",
    "blackforge.recon.evidence",
    "blackforge.recon.materializer",
    "blackforge.recon.capabilities",
    "blackforge.recon.engine",
    "blackforge.webapi",
    "blackforge.webapi.models",
    "blackforge.webapi.mock",
    "blackforge.webapi.normalization",
    "blackforge.webapi.evidence",
    "blackforge.webapi.capabilities",
    "blackforge.webapi.engine",
    "blackforge.webapi.materializer",
    "blackforge.webapi.redaction",
    "blackforge.auth",
    "blackforge.auth.models",
    "blackforge.auth.redaction",
    "blackforge.auth.transport",
    "blackforge.auth.normalization",
    "blackforge.auth.evidence",
    "blackforge.auth.capabilities",
    "blackforge.auth.materializer",
    "blackforge.auth.engine",
    "blackforge.business_logic",
    "blackforge.business_logic.models",
    "blackforge.business_logic.redaction",
    "blackforge.business_logic.transport",
    "blackforge.business_logic.normalization",
    "blackforge.business_logic.evidence",
    "blackforge.business_logic.materializer",
    "blackforge.business_logic.capabilities",
    "blackforge.business_logic.engine",
]

_import_failures = []
for module in modules:
    try:
        importlib.import_module(module)
    except Exception as e:
        _import_failures.append((module, str(e)))

if _import_failures:
    for mod, err in _import_failures:
        print(f"  FAIL: {mod} — {err}")
    raise RuntimeError(f"Import health check failed: {len(_import_failures)} module(s)")

print(f"Blackforge imports OK ({len(modules)} modules verified).")
print("Business logic module imports: PASS")

---

In [ ]:
import subprocess
import sys

print("Running automated test suite...")
# The LLM/torch-heavy files are excluded: importing the HF provider pulls
# ~2GB of torch memory and can SIGKILL the kernel on CPU runtimes. Those
# tests are validated locally and in the Phase 1 notebook.
result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q", "--tb=short",
        "--ignore=tests/test_huggingface_provider.py",
        "--ignore=tests/test_loader.py",
        "--ignore=tests/test_smoke_real_model.py",
    ],
    capture_output=True, text=True, cwd=str(REPO_DIR),
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:] if result.stderr else "")
    raise RuntimeError(f"pytest failed with exit code {result.returncode}")

print("Automated test suite: PASS")

---

In [ ]:
import os
from pathlib import Path

DBROOT = Path("data/phase8_colab").resolve()
DBROOT.mkdir(parents=True, exist_ok=True)
os.environ["BLACKFORGE_DB_PATH"] = str(DBROOT / "blackforge.db")
os.environ["BLACKFORGE_MEMORY_DB_PATH"] = str(DBROOT / "memory.db")
os.environ["BLACKFORGE_EVIDENCE_DB_PATH"] = str(DBROOT / "evidence.db")
os.environ["BLACKFORGE_WORLD_MODEL_DB_PATH"] = str(DBROOT / "world_model.db")
for _p in (DBROOT / "evidence.db", DBROOT / "world_model.db"):
    _p.unlink(missing_ok=True)

from blackforge.runtime.bootstrap import bootstrap

app = bootstrap()
assert app.healthy(), "Blackforge health check failed"
verification = app.verify()
for key in ("config_loaded", "mission_manager_ready", "capability_registry_ready",
            "memory_ready", "evidence_store_ready", "evidence_memory_link_ready",
            "world_model_ready", "recon_ready", "webapi_ready", "auth_ready",
            "business_logic_ready", "authorization_ready", "model_router_ready"):
    assert verification[key], f"{key} must be True"
assert verification["business_logic_ready"] is True, "business_logic_ready must be True (11 typed capabilities)"
assert len(app.capability_registry.list_capabilities()) == 39

BOOTSTRAP_OK = app.healthy() and bool(verification["business_logic_ready"])

for k, v in verification.items():
    symbol = "PASS" if v else "FAIL"
    print(f"  [{symbol}] {k}")

print("\nBlackforge bootstrap (business_logic_ready, 39 registered capabilities): PASS")

---

In [ ]:
from blackforge.business_logic.models import (
    BusinessLogicMode,
    BusinessLogicRequest,
    BusinessLogicStatus,
)
from blackforge.scope.models import TargetScope, Target, detect_target_type
from blackforge.core.types import RiskLevel, TargetType

def _target(value: str) -> Target:
    return Target(value=value, target_type=detect_target_type(value))

MID = "mission_phase8_bl"
SHOP = "shop.example.com"
ERROR_HOSTS = [
    "throttled.example.com", "unreachable.example.com",
    "slow.example.com", "malformed.example.com", "elsewhere.example.com",
]
ALL_HOSTS = [SHOP] + ERROR_HOSTS
scope = TargetScope(
    mission_id=MID,
    allowed_targets=[_target(t) for t in ALL_HOSTS],
    max_risk_level=RiskLevel.HIGH,
)
IDENTS = ["alice", "bob", "customer", "admin", "warehouse"]
req = BusinessLogicRequest(
    mission_id=MID,
    session_id="ses_phase8_bl",
    scope=scope,
    mode=BusinessLogicMode.ACTIVE,
    test_identities=IDENTS,
    max_observations=2000,
    timeout_seconds=30.0,
)

engine = app.business_logic_engine
assert engine is not None and len(engine.capabilities) == 11
expected = sorted([
    "business_logic.workflow_discovery",
    "business_logic.workflow_modeling",
    "business_logic.state_transition_analysis",
    "business_logic.business_rule_analysis",
    "business_logic.ownership_analysis",
    "business_logic.role_boundary_analysis",
    "business_logic.workflow_consistency_analysis",
    "business_logic.controlled_workflow_replay",
    "business_logic.business_logic_hypothesis",
    "business_logic.business_logic_validation",
    "business_logic.workflow_evidence_collection",
])
ids_seen = sorted(c.capability_id for c in engine.capabilities)
assert ids_seen == expected, (ids_seen, expected)
print("Registered business logic capabilities:", ", ".join(c.capability_id for c in engine.capabilities))

_medium = {"controlled_workflow_replay", "business_logic_hypothesis",
           "business_logic_validation", "workflow_evidence_collection"}
_meta_by_id = {c.capability_id: c.meta() for c in engine.capabilities}
for capability_id in expected:
    meta = _meta_by_id[capability_id]
    risk = meta.risk_level.value
    assert (risk == "medium") == (capability_id.split(".")[-1] in _medium), capability_id
    print(
        f"  {meta.id:<35} risk={risk:<6} mode={meta.mode.value:<7} "
        f"targets={[t.value for t in meta.supported_target_types]}"
    )
    assert meta.world_model, f"{capability_id} must materialize into the world model"
CAPS_OK = True

res = engine.discover_workflows(req, SHOP)
assert res.status == BusinessLogicStatus.SUCCESS
assert res.capability_id == "business_logic.workflow_discovery"
assert res.observations[0].workflow == "order_workflow"
print(f"\nworkflow_discovery -> {res.observations[0].workflow} @ {res.observations[0].host} "
      f"[{len(res.observations[0].state_names)} states / {len(res.observations[0].action_names)} actions]")

---

In [ ]:
from blackforge.evidence.models import EvidenceRelation
from blackforge.world_model.query import RelationshipQuery
from blackforge.world_model.models import EntityType

r_discover = engine.discover_workflows(req, SHOP)
r_model = engine.model_workflow(req, SHOP)
r_trans = engine.analyze_state_transitions(req, SHOP)
r_rules = engine.analyze_business_rules(req, SHOP)
r_owner = engine.analyze_ownership(req, SHOP, test_identities=["alice", "bob"])
r_boundaries = engine.analyze_role_boundaries(req, SHOP, test_identities=["customer"])
r_consistency = engine.check_workflow_consistency(req, SHOP)
r_replay = engine.replay_workflow(
    req, SHOP, actions=["process_payment", "ship_order", "cancel_order"],
    start_state="created",
)
r_hyp = engine.hypothesize_business_logic(req, SHOP)
r_valid = engine.validate_business_logic(req, SHOP)
r_evidence = engine.collect_workflow_evidence(req, SHOP)

pipeline_runs = (r_discover, r_model, r_trans, r_rules, r_owner, r_boundaries,
                 r_consistency, r_replay, r_hyp, r_valid, r_evidence)
for r in pipeline_runs:
    assert r.status == BusinessLogicStatus.SUCCESS, (r.capability_id, r.status)
print("All 11 business logic capabilities executed with status SUCCESS")

# Observed expectations on the mock shop
anomalous = [o for o in r_trans.observations if o.anomalous]
assert len(anomalous) == 1 and anomalous[0].action == "ship_order"
assert anomalous[0].source_state == "created" and anomalous[0].target_state == "shipped"
broken = {o.rule for o in r_rules.observations if o.enforcement == "broken"}
assert "only_paid_orders_ship" in broken
violated = {o.invariant for o in r_consistency.observations if o.status == "violated"}
assert "only_paid_orders_ship" in violated
assert r_replay.observations[0].result.value == "success"
assert {o.hypothesis: o.outcome.value for o in r_hyp.observations}.get("cancel_after_payment") == "supported"
validated = {o.hypothesis for o in r_valid.observations if o.result.value == "validated"}
assert "cancel_after_payment" in validated
print("Anomalous created->shipped path, broken rule, violated invariant, replay, elevation: PASS")

# Every observation evidence row is DERIVED_FROM its run's artifact row.
rel_ok = True
count_obs = 0
for r in pipeline_runs:
    artifact = r.evidence_ids[0]
    for ev_id in r.evidence_ids[1:]:
        rels = app.evidence_store.get_relationships(ev_id)
        ok = any(
            x.relation_type == EvidenceRelation.DERIVED_FROM
            and str(x.target_id) == str(artifact)
            for x in rels
        )
        rel_ok = rel_ok and ok
        count_obs += 1
assert rel_ok and count_obs >= 30, count_obs
print(f"DERIVED_FROM links: {count_obs} observations across {len(pipeline_runs)} artifacts")

# Elevation semantics: consensus status is HYPOTHESIZED at most through
# hypothesis, and only the validation capability elevates to VALIDATED.
from blackforge.evidence.models import EvidenceStatus

_stored = {e.id: e.status for e in app.evidence_store.list(limit=10000)}

def _run_statuses(run):
    return {_stored[ev] for ev in run.evidence_ids[1:] if ev in _stored}

assert EvidenceStatus.VALIDATED in _run_statuses(r_valid), _run_statuses(r_valid)
assert EvidenceStatus.VALIDATED not in _run_statuses(r_hyp), _run_statuses(r_hyp)
assert EvidenceStatus.VALIDATED not in _run_statuses(r_rules), _run_statuses(r_rules)
print("Evidence elevation (consensus VALIDATED only via business_logic.validation): PASS")

wm = engine.world_model
workflow = wm.find_entity(MID, EntityType.WORKFLOW, "order_workflow", namespace=SHOP)
state = wm.find_entity(MID, EntityType.BUSINESS_STATE, "paid", namespace=SHOP)
action = wm.find_entity(MID, EntityType.BUSINESS_ACTION, "ship_order", namespace=SHOP)
alice = wm.find_entity(MID, EntityType.IDENTITY, "alice", namespace=SHOP)
role = wm.find_entity(MID, EntityType.ROLE, "customer", namespace=SHOP)
permission = wm.find_entity(MID, EntityType.PERMISSION, "submit_order::orders", namespace=SHOP)
resource = wm.find_entity(MID, EntityType.RESOURCE, "orders", namespace=SHOP)
assert workflow is not None and state is not None and action is not None
assert alice is not None and role is not None and permission is not None and resource is not None
print("WORKFLOW:", workflow.name, "| STATE:", state.name, "| ACTION:", action.name)
print("IDENTITY:", alice.name, "| ROLE:", role.name, "| PERMISSION:", permission.name, "| RESOURCE:", resource.name)

rels = wm.list_relationships(RelationshipQuery(mission_id=MID, limit=1000))
rel_types = {getattr(r.relationship_type, "value", r.relationship_type) for r in rels}
expected_types = {"has_workflow", "has_state", "has_action", "transitions_to",
                  "operates_on", "belongs_to", "has_permission", "applies_to"}
assert expected_types <= rel_types, rel_types
print("Relationship types:", ", ".join(sorted(rel_types)))
attack_graph = rel_types & {"exploits", "can_compromise", "leads_to", "enables"}
assert not attack_graph, f"Attack-graph relationships must not be materialized: {attack_graph}"
print("No attack-graph relationship types (EXPLOITS/CAN_COMPROMISE/LEADS_TO/ENABLES): PASS")

# Canonical entity revisioning: each new workflow REPORT supersedes the
# previous revision, and the assertion ledger stays attached to the revision
# that produced it. Aggregate across the active + superseded revisions.
from blackforge.world_model.query import WorldQuery
from blackforge.world_model.models import WorldLifecycle

_wf_q = lambda lc: WorldQuery(mission_id=MID, entity_type=EntityType.WORKFLOW,
                              lifecycle=lc, limit=20)
_workflow_entities = wm.list_entities(_wf_q(WorldLifecycle.ACTIVE))
_workflow_entities += wm.list_entities(_wf_q(WorldLifecycle.SUPERSEDED))
_workflow_entities += wm.list_entities(_wf_q(WorldLifecycle.ARCHIVED))
assert len(_workflow_entities) >= 1
assertions = [a for _e in _workflow_entities for a in wm.list_assertions(str(_e.id))]
prop_prefixes = {a.property_key.split(".")[0] for a in assertions}
assert {"rule", "invariant", "replay", "hypothesis", "validation"} <= prop_prefixes, prop_prefixes
assert any("transition_violation" in a.property_key for a in assertions)
statuses = {a.epistemic_status.value for a in assertions}
assert "validated" in statuses, statuses
print(f"Workflow assertions: {len(assertions)} across {len(_workflow_entities)} entity revision(s) "
      f"(prefixes {sorted(prop_prefixes)}, statuses {sorted(statuses)})")

---

In [ ]:
# Mission isolation: business logic work under a second mission is disjoint.
MID2 = "mission_phase8_bl_other"
scope2 = TargetScope(
    mission_id=MID2,
    allowed_targets=[_target(SHOP)],
    max_risk_level=RiskLevel.HIGH,
)
req2 = BusinessLogicRequest(
    mission_id=MID2, session_id="ses_phase8_bl_2", scope=scope2,
    mode=BusinessLogicMode.ACTIVE, test_identities=["alice"],
)
res2 = engine.discover_workflows(req2, SHOP)
other_ids = {str(x) for x in res2.evidence_ids}
assert other_ids.isdisjoint({str(x) for x in r_discover.evidence_ids})
assert app.evidence_store.count(MID2) == len(res2.evidence_ids)
assert engine.world_model.count_entities(MID2) >= 1
print("Mission isolation: second mission produced its own evidence/world rows: PASS")

# Redaction: credential-like fields are replaced by a stable literal marker.
from blackforge.business_logic.redaction import (
    credential_value_redacted,
    redact_credential_fields,
    redact_nested_credential_values,
)

doc = {
    "note": "keep me",
    "credential_value": "supersecret",
    "nested": {"token": "abc123", "access_token": "xyz", "keep": 1},
}
clean = redact_credential_fields(doc)
assert clean["note"] == "keep me"
assert clean["credential_value"] == "REDACTED"
assert clean["nested"]["token"] == "REDACTED"
assert clean["nested"]["access_token"] == "REDACTED"
clean2 = redact_nested_credential_values({"meta": {"credential_value": "s"}, "keep": "v"})
assert clean2["meta"]["credential_value"] == "REDACTED" and clean2["keep"] == "v"
assert credential_value_redacted() == "REDACTED"
print("Redaction boundary (credential-like fields -> literal REDACTED marker): PASS")

---

In [ ]:
from blackforge.core.errors import (
    AuthorizationError,
    BusinessLogicExecutionError,
)
from blackforge.business_logic.models import BusinessLogicStatus

# 1) Target outside the scope is denied BEFORE any transport runs.
denied_out = True
try:
    engine.discover_workflows(req, "scanme.example.org")
    denied_out = False
except AuthorizationError:
    pass
assert denied_out
print("Out-of-scope target denied before transport execution: PASS")

# 2) Unknown capability is rejected (no generic execution surface).
unknown_rejected = True
try:
    engine.run(req, "business_logic.not_real", SHOP)
    unknown_rejected = False
except BusinessLogicExecutionError:
    pass
assert unknown_rejected
print("Unknown capability rejected (no generic execution surface): PASS")

# 3) Ownership / role-boundary never guess identities.
no_ids_req = BusinessLogicRequest(
    mission_id=MID, session_id="s_noid", scope=scope,
    mode=BusinessLogicMode.ACTIVE, test_identities=[], max_observations=2000,
)
missing_ids = True
try:
    engine.analyze_ownership(no_ids_req, SHOP)
    missing_ids = False
except BusinessLogicExecutionError as e:
    assert "explicit authorized test identities" in str(e)
assert missing_ids
bad_ids = True
try:
    engine.analyze_ownership(req, SHOP, test_identities=["mallory"])
    bad_ids = False
except BusinessLogicExecutionError as e:
    assert "not authorized" in str(e)
assert bad_ids
print("Explicit-identity enforcement (no guessing, no unauthorized identities): PASS")

# 4) Replay is fail-closed: PROHIBITED actions are refused before transport.
prohibited = True
try:
    engine.replay_workflow(req, SHOP, actions=["refund_order"], start_state="created")
    prohibited = False
except BusinessLogicExecutionError as e:
    assert "PROHIBITED" in str(e)
assert prohibited
print("Fail-closed replay safety (PROHIBITED action refused): PASS")

# 5) Failure states on the mock error hosts.
assert engine.discover_workflows(req, "throttled.example.com").status == BusinessLogicStatus.RATE_LIMITED
assert engine.discover_workflows(req, "unreachable.example.com").status == BusinessLogicStatus.REQUEST_FAILED
assert engine.discover_workflows(req, "malformed.example.com").status == BusinessLogicStatus.MALFORMED_RESPONSE
assert engine.discover_workflows(req, "slow.example.com").status == BusinessLogicStatus.TIMEOUT
limited_req = BusinessLogicRequest(
    mission_id=MID, session_id="s_lim", scope=scope,
    mode=BusinessLogicMode.ACTIVE, test_identities=IDENTS, max_observations=1,
)
limited = engine.analyze_business_rules(limited_req, SHOP)
assert limited.status == BusinessLogicStatus.LIMITED and len(limited.observations) == 1
assert any("limit" in w for w in limited.warnings)
print("Failure states (RATE_LIMITED, REQUEST_FAILED, MALFORMED_RESPONSE, TIMEOUT, LIMITED): PASS")

# 6) Elevation bookkeeping: hypothesis evidence stays HYPOTHESIZED,
#    only the validation capability writes VALIDATED observation rows
#    (artifact rows can never carry VALIDATED status).
rows = {e.id: e.status for e in app.evidence_store.list(limit=10000)}
hyp_rows = [
    e for e in app.evidence_store.list(limit=10000)
    if e.mission_id == MID and e.status == EvidenceStatus.HYPOTHESIZED
]
assert hyp_rows, "hypothesis evidence must persist as HYPOTHESIZED"
assert rows[r_valid.evidence_ids[0]] != EvidenceStatus.VALIDATED, "artifacts never VALIDATED"
v_obs = [rows.get(x) for x in r_valid.evidence_ids[1:]]
assert EvidenceStatus.VALIDATED in v_obs, v_obs
h_obs = [rows.get(x) for x in r_hyp.evidence_ids[1:]]
assert EvidenceStatus.VALIDATED not in h_obs, h_obs
print("Evidence elevation (HYPOTHESIZED records, artifacts never VALIDATED): PASS")

---

In [ ]:
from blackforge.evidence.repository import SQLiteEvidenceRepository
from blackforge.evidence.store import EvidenceStore
from blackforge.world_model.repository import SQLiteWorldRepository
from blackforge.world_model.store import WorldModelStore

# Fresh connections over the same SQLite files prove restart persistence.
fresh_ev = EvidenceStore(SQLiteEvidenceRepository(str(DBROOT / "evidence.db")))
fresh_wm = WorldModelStore(SQLiteWorldRepository(str(DBROOT / "world_model.db")))

persisted_ev = fresh_ev.count(MID) == app.evidence_store.count(MID)
persisted_wm = fresh_wm.count_entities(MID) == engine.world_model.count_entities(MID)
workflow_entity = fresh_wm.find_entity(MID, EntityType.WORKFLOW, "order_workflow", namespace=SHOP)
alice_entity = fresh_wm.find_entity(MID, EntityType.IDENTITY, "alice", namespace=SHOP)
persisted_workflow = workflow_entity is not None
persisted_alice = alice_entity is not None
assert persisted_ev and persisted_wm and persisted_workflow and persisted_alice
rel_count = len(fresh_wm.list_relationships(RelationshipQuery(mission_id=MID, limit=1000)))
assert rel_count > 0
assert fresh_ev.count(MID) > 0
PERSIST_OK = persisted_ev and persisted_wm and persisted_workflow and persisted_alice

for store in (fresh_ev, fresh_wm):
    store.close()
try:
    app.evidence_store.close()
except Exception:
    pass
try:
    app.world_model.close()
except Exception:
    pass
print("Restart persistence (fresh connections on same DB files): PASS")
print("Backends closed. Validation summary below.")

---

In [ ]:
results = {}
phase_checks = {
    "repository_integrity": (REPO_DIR / "blackforge" / "business_logic" / "engine.py").exists(),
    "phase8_modules": bool(
        (REPO_DIR / "blackforge" / "business_logic" / "capabilities.py").exists()
        and (REPO_DIR / "blackforge" / "business_logic" / "evidence.py").exists()
        and (REPO_DIR / "blackforge" / "business_logic" / "materializer.py").exists()
        and (REPO_DIR / "blackforge" / "business_logic" / "redaction.py").exists()
    ),
    "imports": len(_import_failures) == 0,
    "bootstrap_business_logic_ready": BOOTSTRAP_OK,
    "capability_surface": CAPS_OK,
    "pipeline_evidence": rel_ok,
    "world_materialized": workflow is not None and state is not None and action is not None,
    "identity_boundary": alice is not None and role is not None and permission is not None and resource is not None,
    "assertions_bound": len(assertions) > 0 and "validated" in statuses,
    "no_attack_graph": not bool(attack_graph),
    "scope_authorization": denied_out,
    "unknown_capability_rejected": unknown_rejected,
    "test_identities_required": missing_ids and bad_ids,
    "replay_fail_closed": prohibited,
    "mission_isolation": bool(other_ids.isdisjoint({str(x) for x in r_discover.evidence_ids})),
    "restart_persistence": PERSIST_OK,
}

# The pytest cell aborts the run on failure, so reaching this cell proves it passed.
pytest_passed = True
install_ok = len(_import_failures) == 0

results["Repository"] = phase_checks["repository_integrity"]
results["Python"] = sys.version_info >= (3, 10)
results["Hardware"] = True  # CPU fallback always works; this notebook needs no GPU
results["Installation"] = install_ok
results["Imports"] = install_ok
results["Automated tests"] = pytest_passed
results["Bootstrap"] = phase_checks["bootstrap_business_logic_ready"]
results["Phase-specific tests"] = all(phase_checks.values())
results["Security checks"] = (
    phase_checks["scope_authorization"]
    and phase_checks["unknown_capability_rejected"]
    and phase_checks["test_identities_required"]
    and phase_checks["replay_fail_closed"]
    and phase_checks["no_attack_graph"]
)

print()
print("=" * 60)
print("PHASE 8 COLAB VALIDATION SUMMARY")
print("=" * 60)
for name, ok in results.items():
    symbol = "PASS" if ok else "FAIL"
    print(f"  [{symbol}] {name}")

_all_ok = all(results.values()) and all(phase_checks.values())
assert _all_ok, "One or more validation checks failed"

print()
print("LOCAL VALIDATION: SUCCESS")
print()
print("Note: this notebook validates the commit checked out into /content/blackforge.")

---